# CSCN8000 – Artificial Intelligence Algorithms and Mathematics

## Assignment 1 – Parkinson's Disease Classification

**Student Name:** Preethi Vasudevan Nair  
**Student ID:** 9125985

### Objective

In this assignment, I will use machine learning to predict Parkinson's disease based on patient speech measurements.

I will prepare the dataset, train four classification models, and compare their performance using accuracy, confusion matrices, and classification reports. I will also test the models using an unseen sample and save the predictions in a CSV file.

## Question 1 – Data Preprocessing

### (i) Data Analysis

In this section, I will load the Parkinson's disease dataset and explore its structure. I will check the number of rows and columns, view the first few records, and understand the features available for classification.

In [1]:
import pandas as pd

# Load the dataset using the second row as column names
df = pd.read_csv("./data/pd_speech_features.csv", header=1)

# Display the dataset dimensions
print("Dataset Shape:", df.shape)

# Display the first five records
display(df.head())

Dataset Shape: (756, 755)


,id,gender,PPE,DFA,RPDE,numPulses,numPeriodsPulses,meanPeriodPulses,stdDevPeriodPulses,locPctJitter,...,tqwt_kurtosisValue_dec_28,tqwt_kurtosisValue_dec_29,tqwt_kurtosisValue_dec_30,tqwt_kurtosisValue_dec_31,tqwt_kurtosisValue_dec_32,tqwt_kurtosisValue_dec_33,tqwt_kurtosisValue_dec_34,tqwt_kurtosisValue_dec_35,tqwt_kurtosisValue_dec_36,class
0,0,1,0.85247,0.71826,0.57227,240,239,0.008064,0.000087,0.00218,...,1.5620,2.6445,3.8686,4.2105,5.1221,4.4625,2.6202,3.0004,18.9405,1
1,0,1,0.76686,0.69481,0.53966,234,233,0.008258,0.000073,0.00195,...,1.5589,3.6107,23.5155,14.1962,11.0261,9.5082,6.5245,6.3431,45.1780,1
2,0,1,0.85083,0.67604,0.58982,232,231,0.008340,0.000060,0.00176,...,1.5643,2.3308,9.4959,10.7458,11.0177,4.8066,2.9199,3.1495,4.7666,1
3,1,0,0.41121,0.79672,0.59257,178,177,0.010858,0.000183,0.00419,...,3.7805,3.5664,5.2558,14.0403,4.2235,4.6857,4.8460,6.2650,4.0603,1
4,1,0,0.32790,0.79782,0.53028,236,235,0.008162,0.002669,0.00535,...,6.1727,5.8416,6.0805,5.7621,7.7817,11.6891,8.2103,5.0559,6.1164,1


In [2]:
# Check column names and data types
print("Dataset Information:")
df.info()

# Check missing values
print("\nTotal Missing Values:", df.isnull().sum().sum())

# Check duplicate rows
print("Duplicate Rows:", df.duplicated().sum())

# Check target class distribution
print("\nClass Distribution:")
print(df["class"].value_counts())

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 756 entries, 0 to 755
Columns: 755 entries, id to class
dtypes: float64(749), int64(6)
memory usage: 4.4 MB

Total Missing Values: 0
Duplicate Rows: 1

Class Distribution:
class
1    564
0    192
Name: count, dtype: int64


In [3]:
# Check how many unique patient IDs exist
print("\nUnique Patient IDs:", df["id"].nunique())

# Check whether IDs appear multiple times
print("\nPatient ID Counts:")
print(df["id"].value_counts().head(10))


Unique Patient IDs: 252

Patient ID Counts:
id
0    3
1    3
2    3
3    3
4    3
5    3
6    3
7    3
8    3
9    3
Name: count, dtype: int64


In [4]:
# Check the duplicate records
duplicate_records = df[df.duplicated(keep=False)]

print("Number of duplicate rows:", df.duplicated().sum())

display(duplicate_records[["id", "class"]])

Number of duplicate rows: 1


,id,class
111,37,1
113,37,1


### Observations (Data Analysis)

The dataset contains 756 records and 755 columns. I checked the data types, missing values, duplicate records, and class distribution.

There are no missing values, but I found one duplicate record belonging to patient ID 37. The dataset contains 252 unique patients, with three recordings per patient.

The target column is `class`, where 1 represents Parkinson's disease and 0 represents no Parkinson's disease. There are 564 records in class 1 and 192 records in class 0.

This shows that the dataset is imbalanced because there are more Parkinson's records than non-Parkinson's records.

### (ii) Data Preprocessing

In this section, I will remove duplicate records and verify that the cleaned dataset has no missing values or duplicates.

In [5]:
# Remove duplicate records
df_cleaned = df.drop_duplicates().copy()

# Verify the cleaned dataset
print("Original Dataset Shape:", df.shape)
print("Cleaned Dataset Shape:", df_cleaned.shape)
print("Remaining Duplicate Rows:", df_cleaned.duplicated().sum())

# Check missing values after cleaning
print("Missing Values:", df_cleaned.isnull().sum().sum())

Original Dataset Shape: (756, 755)
Cleaned Dataset Shape: (755, 755)
Remaining Duplicate Rows: 0
Missing Values: 0


### Observation (Data Preprocessing)

During data analysis, I found one duplicate record belonging to patient ID 37. I removed the duplicate to avoid using the same observation twice.

The cleaned dataset contains 755 records and 755 columns. There are no missing values, so no missing-value imputation is needed.

The `id` column identifies patients and will not be used as an input feature for model training.

### (iii) Data Standardization

In this section, I will standardize the numerical features so they are on a similar scale. This is important because models such as KNN and SVM are sensitive to differences in feature values.

Before standardization, I will split the dataset into training and testing sets. I will fit the StandardScaler using only the training data and then apply the same transformation to the testing data to avoid data leakage.

In [6]:
from sklearn.model_selection import GroupShuffleSplit

# Separate the input features and target column
X = df_cleaned.drop(columns=["id", "class"])
y = df_cleaned["class"]

# Use patient IDs to keep recordings from the same patient together
groups = df_cleaned["id"]

# Split patients into approximately 80% training and 20% testing
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

# Verify that patients are not shared between training and testing
train_patients = set(groups.iloc[train_idx])
test_patients = set(groups.iloc[test_idx])

print("Patients in both sets:", len(train_patients.intersection(test_patients)))

Training Samples: 602
Testing Samples: 153
Patients in both sets: 0


In [7]:
from sklearn.preprocessing import StandardScaler

# Create the standard scaler
scaler = StandardScaler()

# Fit the scaler on training data and standardize it
X_train_scaled = scaler.fit_transform(X_train)

# Standardize testing data using the same scaler
X_test_scaled = scaler.transform(X_test)

# Display the standardized data
print("Standardized Training Data Shape:", X_train_scaled.shape)
print("Standardized Testing Data Shape:", X_test_scaled.shape)

# Display the first five standardized values
print("\nFirst 5 Standardized Values:")
print(X_train_scaled[0][:5])

Standardized Training Data Shape: (602, 753)
Standardized Testing Data Shape: (153, 753)

First 5 Standardized Values:
[ 0.92629622  0.62419405  0.26191131  0.57418533 -0.81410942]


### Observation (Data Standardization)

I used StandardScaler to standardize the numerical features. The scaler was fitted only on the training data and then applied to the testing data to avoid data leakage.

After standardization, the training dataset contains 602 records and 753 features, while the testing dataset contains 153 records and 753 features.

Standardization helps bring features onto a comparable scale, which is especially important for KNN and SVM.

## Question 2 – Use 80% of Samples as Training Data

I divided the cleaned dataset into approximately 80% training data and 20% testing data using GroupShuffleSplit.

The training dataset contains 602 samples (79.74%), and the testing dataset contains 153 samples (20.26%).

I used patient IDs to keep recordings from the same patient together. The verification showed that no patient appears in both datasets, helping prevent data leakage.

In [8]:
# Verify the training and testing percentages
total_samples = len(df_cleaned)

train_percentage = len(X_train) / total_samples * 100
test_percentage = len(X_test) / total_samples * 100

print(f"Training Samples: {len(X_train)} ({train_percentage:.2f}%)")
print(f"Testing Samples: {len(X_test)} ({test_percentage:.2f}%)")
print(f"Total Samples: {total_samples}")

Training Samples: 602 (79.74%)
Testing Samples: 153 (20.26%)
Total Samples: 755


## Question 3 – Model Training

In this question, I will train four machine learning models to predict Parkinson's disease: KNN, Random Forest, SVM, and Decision Tree.

I will use the training data prepared in the previous steps. For KNN and SVM, I will use the standardized features because these models are sensitive to feature scales.

After training the models, I will use the testing data to compare their performance.

### (i) K-Nearest Neighbors (KNN)

KNN predicts the class of a new sample by looking at the closest samples in the training data.

I will start with K = 5, which means the model will consider the five nearest neighbors when making a prediction.

In [9]:
from sklearn.neighbors import KNeighborsClassifier

# Create the KNN model with 5 neighbors
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train the model using standardized training data
knn_model.fit(X_train_scaled, y_train)

print("KNN Model Training Completed")

KNN Model Training Completed


### (ii) Random Forest

Random Forest is a machine learning algorithm that uses multiple decision trees to make predictions. Each tree makes its own prediction, and the model combines their results to decide the final class.

I will train a Random Forest model using 100 decision trees. I will use the original training features because Random Forest does not require standardization.

In [10]:
from sklearn.ensemble import RandomForestClassifier

# Create the Random Forest model with 100 trees
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model using the original training data
rf_model.fit(X_train, y_train)

print("Random Forest Model Training Completed")

Random Forest Model Training Completed


### (iii) Support Vector Machine (SVM)

SVM is a classification algorithm that tries to find a boundary between different classes.

I will use SVM to classify the patient records as Parkinson's or non-Parkinson's. I will use the standardized training data because SVM is sensitive to differences in feature scales.

In [11]:
from sklearn.svm import SVC

# Create the SVM model
svm_model = SVC(
    kernel="rbf",
    random_state=42
)

# Train the model using standardized training data
svm_model.fit(X_train_scaled, y_train)

print("SVM Model Training Completed")

SVM Model Training Completed


### (iv) Decision Tree

Decision Tree is a classification algorithm that makes predictions by following a series of decisions based on feature values.

I will train a Decision Tree model using the original training data. Since Decision Tree does not depend on distances between samples, standardization is not required.

In [12]:
from sklearn.tree import DecisionTreeClassifier

# Create the Decision Tree model
dt_model = DecisionTreeClassifier(
    random_state=42
)

# Train the model using the original training data
dt_model.fit(X_train, y_train)

print("Decision Tree Model Training Completed")

Decision Tree Model Training Completed


## Question 4 – Predicting Parkinson's Disease on Test Data

In this question, I will use the four trained models to predict Parkinson's disease on the testing dataset.

I will use the standardized testing data for KNN and SVM, and the original testing data for Random Forest and Decision Tree.

Each model will predict either class 0 (no Parkinson's disease) or class 1 (Parkinson's disease). I will compare these predictions with the actual classes when evaluating the models.

In [13]:
# Predict the test data using all four trained models

knn_predictions = knn_model.predict(X_test_scaled)

rf_predictions = rf_model.predict(X_test)

svm_predictions = svm_model.predict(X_test_scaled)

dt_predictions = dt_model.predict(X_test)

print("Predictions completed for all four models!")

Predictions completed for all four models!


In [14]:
# Compare the actual classes with the predictions
predictions_df = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "KNN": knn_predictions,
    "Random Forest": rf_predictions,
    "SVM": svm_predictions,
    "Decision Tree": dt_predictions
})

print("First 10 Test Predictions:")
display(predictions_df.head(10))

print("\nTotal Predictions:", len(predictions_df))

First 10 Test Predictions:


,Actual,KNN,Random Forest,SVM,Decision Tree
0,1,1,1,1,1
1,1,1,1,1,1
2,1,1,1,1,1
3,1,1,1,1,1
4,1,0,1,1,1
5,1,1,1,1,1
6,1,1,1,1,1
7,1,1,1,1,1
8,1,1,1,1,1
9,1,1,1,1,1



Total Predictions: 153


### Observation (Test Data Predictions)

I used all four trained models to predict Parkinson's disease for the 153 test samples.

I compared the predicted classes with the actual classes in a DataFrame. From the first 10 records, I noticed that the models do not always give the same prediction.

In the next steps, I will evaluate the models to find out which one performs better.

## Question 5 – Predicting Parkinson's Disease for a New Sample

In this question, I will test the trained models using one patient sample from the testing dataset.

I will use the same sample for all four models and compare their predictions with the actual class. This will help me check whether the models can predict a sample they have not seen during training.

In [15]:
# Select one unseen sample from the test dataset
new_sample = X_test.iloc[[0]]
new_sample_scaled = X_test_scaled[[0]]

# Get the actual class
actual_class = y_test.iloc[0]

# Predict using all four models
new_predictions = {
    "KNN": knn_model.predict(new_sample_scaled)[0],
    "Random Forest": rf_model.predict(new_sample)[0],
    "SVM": svm_model.predict(new_sample_scaled)[0],
    "Decision Tree": dt_model.predict(new_sample)[0]
}

# Display the results
print("Actual Class:", actual_class)

print("\nPredictions for the New Sample:")
for model_name, prediction in new_predictions.items():
    print(f"{model_name}: {prediction}")

Actual Class: 1

Predictions for the New Sample:
KNN: 1
Random Forest: 1
SVM: 1
Decision Tree: 1


### Observation (New Sample Prediction)

I selected one sample from the testing dataset to check how the four trained models make predictions on unseen data.

The actual class of this sample was 1, which represents Parkinson's disease. All four models predicted class 1, so their predictions were correct for this sample.

This shows that the models were able to classify this particular test sample correctly. However, I still need to evaluate all the test samples to understand their overall performance.

## Question 6 – Model Evaluation

In this question, I will evaluate the performance of all four machine learning models using the testing dataset.

I will calculate the accuracy of each model and compare the results to find which model performs best. I will also use confusion matrices and classification reports to understand how well the models predict both classes.

In [16]:
from sklearn.metrics import accuracy_score
# Calculate accuracy for all four models
knn_accuracy = accuracy_score(y_test, knn_predictions)
rf_accuracy = accuracy_score(y_test, rf_predictions)
svm_accuracy = accuracy_score(y_test, svm_predictions)
dt_accuracy = accuracy_score(y_test, dt_predictions)

# Store accuracies in a DataFrame
accuracy_df = pd.DataFrame({
    "Model": ["KNN", "Random Forest", "SVM", "Decision Tree"],
    "Accuracy": [
        knn_accuracy,
        rf_accuracy,
        svm_accuracy,
        dt_accuracy
    ]
})

# Display accuracy as a percentage
accuracy_df["Accuracy (%)"] = (
    accuracy_df["Accuracy"] * 100
).round(2)

print("Model Accuracy Comparison:")
display(accuracy_df)

Model Accuracy Comparison:


,Model,Accuracy,Accuracy (%)
0,KNN,0.797386,79.74
1,Random Forest,0.823529,82.35
2,SVM,0.810458,81.05
3,Decision Tree,0.745098,74.51


### Observation (Model Accuracy Comparison)

I calculated the accuracy of all four models using the testing dataset.

Random Forest achieved the highest accuracy of 82.35%, followed by SVM with 81.05% and KNN with 79.74%. Decision Tree had the lowest accuracy of 74.51%.

Based on these results, Random Forest performed the best among the four models in terms of accuracy.

However, since the dataset is imbalanced, I will also check the confusion matrices and classification reports to understand how well each model predicts both classes.

### Confusion Matrix and Classification Report

After comparing the accuracy of the four models, I will check their confusion matrices and classification reports.

I will use the confusion matrices to see how many predictions were correct and incorrect for each class. The classification reports will help me compare precision, recall, and F1-score for Parkinson's and non-Parkinson's records.

In [17]:
from sklearn.metrics import confusion_matrix, classification_report

# Store predictions from all four models
model_predictions = {
    "KNN": knn_predictions,
    "Random Forest": rf_predictions,
    "SVM": svm_predictions,
    "Decision Tree": dt_predictions
}

# Display confusion matrix and classification report for each model
for model_name, predictions in model_predictions.items():

    print(f"\n{'=' * 45}")
    print(f"Model: {model_name}")
    print(f"{'=' * 45}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, predictions, labels=[0, 1]))

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["No Parkinson's", "Parkinson's"],
        zero_division=0
    ))


Model: KNN

Confusion Matrix:
[[ 18  24]
 [  7 104]]

Classification Report:
                precision    recall  f1-score   support

No Parkinson's       0.72      0.43      0.54        42
   Parkinson's       0.81      0.94      0.87       111

      accuracy                           0.80       153
     macro avg       0.77      0.68      0.70       153
  weighted avg       0.79      0.80      0.78       153


Model: Random Forest

Confusion Matrix:
[[ 17  25]
 [  2 109]]

Classification Report:
                precision    recall  f1-score   support

No Parkinson's       0.89      0.40      0.56        42
   Parkinson's       0.81      0.98      0.89       111

      accuracy                           0.82       153
     macro avg       0.85      0.69      0.72       153
  weighted avg       0.84      0.82      0.80       153


Model: SVM

Confusion Matrix:
[[ 16  26]
 [  3 108]]

Classification Report:
                precision    recall  f1-score   support

No Parkinson's       

In [18]:
# Display confusion matrices in a table
for model_name, predictions in model_predictions.items():
    cm = confusion_matrix(y_test, predictions, labels=[0, 1])

    print(f"\n{model_name}")
    display(pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Predicted 0", "Predicted 1"]
    ))


KNN


,Predicted 0,Predicted 1
Actual 0,18,24
Actual 1,7,104



Random Forest


,Predicted 0,Predicted 1
Actual 0,17,25
Actual 1,2,109



SVM


,Predicted 0,Predicted 1
Actual 0,16,26
Actual 1,3,108



Decision Tree


,Predicted 0,Predicted 1
Actual 0,23,19
Actual 1,20,91


### Final Observation (Model Evaluation)

I evaluated all four models using accuracy, confusion matrices, and classification reports.

Random Forest achieved the highest accuracy of 82.35%, followed by SVM with 81.05%, KNN with 79.74%, and Decision Tree with 74.51%.

From the confusion matrices, I found that Random Forest correctly predicted 109 Parkinson's records and missed only 2. SVM missed 3 Parkinson's records, KNN missed 7, and Decision Tree missed 20.

I also noticed that the models performed better at identifying Parkinson's records than non-Parkinson's records. This may be related to the class imbalance in the dataset.

Based on the test accuracy, I selected Random Forest as the best-performing model for this assignment. However, the confusion matrices and classification reports show that it still makes incorrect predictions, especially for non-Parkinson's records.

### Observation (Classification Report)

From the classification reports, I noticed that the models performed better at identifying Parkinson's records than non-Parkinson's records.

Random Forest correctly identified 109 out of 111 Parkinson's records, giving it a recall of approximately 98.20% for class 1. SVM also performed well, correctly identifying 108 out of 111 records.

However, the models had lower recall for class 0, which means they had more difficulty identifying patients without Parkinson's disease.

The precision, recall, and F1-scores helped me understand the models' performance beyond accuracy. Since the dataset has more Parkinson's records, I found it important to check how well the models predicted both classes.

## Question 7 – Save Test Data Predictions to CSV

In this question, I will save the predictions from all four models into a single CSV file.

The file will contain the actual class and the predicted classes from KNN, Random Forest, SVM, and Decision Tree for all 153 test samples.

This will make it easier to compare the predictions and review the results outside the notebook.

In [19]:
# Save all model predictions to a single CSV file
predictions_df.to_csv(
    "Parkinsons_Test_Predictions.csv",
    index=False
)

print("Predictions saved successfully!")

# Verify the saved CSV file
saved_predictions = pd.read_csv("Parkinsons_Test_Predictions.csv")

print("\nSaved CSV Shape:", saved_predictions.shape)

display(saved_predictions.head())

Predictions saved successfully!

Saved CSV Shape: (153, 5)


,Actual,KNN,Random Forest,SVM,Decision Tree
0,1,1,1,1,1
1,1,1,1,1,1
2,1,1,1,1,1
3,1,1,1,1,1
4,1,0,1,1,1


### Observation (Saving Test Predictions)

I saved the test predictions from all four models into a single CSV file named `Parkinsons_Test_Predictions.csv`.

After saving the file, I loaded it again to verify the results. The CSV contains 153 rows and 5 columns, including the actual class and the predictions from KNN, Random Forest, SVM, and Decision Tree.

The file was saved successfully, and I can use it to compare the predictions from all four models.